# Torch Training Server Example

This notebook demonstrates how to use the FastAPI-based PyTorch training server and client for efficient tensor transmission over HTTP.


## Quick Start

To run the complete test suite, use the test script:


In [2]:
import modal

In [3]:
app_obj = modal.App.lookup("tinkerbell-service", environment_name="main")

In [12]:
f = modal.Function.from_name("tinkerbell-service", "deploy_on_modal.<locals>.serve")

/var/folders/lz/f4qx65kx5dn_5mxblzh2s5h00000gn/T/ipykernel_78930/1324908601.py:1: DeprecationError: 2025-02-11: Looking up class methods using Function.from_name will be deprecated in a future version of Modal.
Use modal.Cls.from_name instead, e.g.

deploy_on_modal = modal.Cls.from_name("tinkerbell-service", "deploy_on_modal")
instance = deploy_on_modal(...)
instance.<locals>.serve.remote(...)

  f = modal.Function.from_name("tinkerbell-service", "deploy_on_modal.<locals>.serve")


In [14]:
f.get_web_url()

'https://jesterlabs--training-service.modal.run'

In [21]:
import torch
import torch.nn.functional as F

In [22]:
params = torch.randn(10, 10, requires_grad=True)

In [ ]:
out = F.mse_loss(params, torch.randn(10, 10, requires_grad=False))

In [16]:
out.backward()

In [20]:
params.grad.shape

torch.Size([10, 10])

In [1]:
# Create model
from transformers import AutoConfig, AutoModelForCausalLM

config = AutoConfig.from_pretrained("Qwen/Qwen3-0.6B")
config.n_layer = 4  # Small model for demo
model = AutoModelForCausalLM.from_config(config)


# Apply tensor parallelism
for name, module in model.named_modules():
    print(f"{name}: {module}")

/Users/shyam/anaconda3/envs/py312/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

: Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): Q

In [2]:
parallelize_plan = {
    # Embedding layer
    "model.embed_tokens": ["column"],
    
    # Attention projections (all layers)
    "model.layers.*.self_attn.q_proj": ["column"],
    "model.layers.*.self_attn.k_proj": ["column"],
    "model.layers.*.self_attn.v_proj": ["column"],
    "model.layers.*.self_attn.o_proj": ["row"],
    
    # MLP projections (all layers)
    "model.layers.*.mlp.gate_proj": ["column"],
    "model.layers.*.mlp.up_proj": ["column"],
    "model.layers.*.mlp.down_proj": ["row"],
    
    # LM head
    "lm_head": ["column"],
}

In [3]:
test = "model.layers.*.self_attn.q_proj"

In [4]:
model = AutoModelForCausalLM.from_config(config)

In [19]:
import re
import fnmatch

def get_submodules_with_wildcard(model, pattern):
    """
    Get all submodules matching a wildcard pattern.
    
    Args:
        model: PyTorch model
        pattern: Pattern with wildcards (e.g., "model.layers.*.self_attn.q_proj")
    
    Returns:
        List of (name, module) tuples matching the pattern
    """
    # Convert wildcard pattern to regex
    regex_pattern = fnmatch.translate(pattern)
    regex = re.compile(regex_pattern)
    
    matching_modules = []
    for name, _ in model.named_modules():
        if regex.match(name):
            matching_modules.append(name)
    
    return matching_modules

In [20]:
from torch.distributed.tensor.parallel import SequenceParallel


In [16]:
submodules_patterns = list(parallelize_plan.keys())

In [17]:
get_submodules_with_wildcard(model, submodules_patterns[1])

[('model.layers.0.self_attn.q_proj',
  Linear(in_features=1024, out_features=2048, bias=False)),
 ('model.layers.1.self_attn.q_proj',
  Linear(in_features=1024, out_features=2048, bias=False)),
 ('model.layers.2.self_attn.q_proj',
  Linear(in_features=1024, out_features=2048, bias=False)),
 ('model.layers.3.self_attn.q_proj',
  Linear(in_features=1024, out_features=2048, bias=False)),
 ('model.layers.4.self_attn.q_proj',
  Linear(in_features=1024, out_features=2048, bias=False)),
 ('model.layers.5.self_attn.q_proj',
  Linear(in_features=1024, out_features=2048, bias=False)),
 ('model.layers.6.self_attn.q_proj',
  Linear(in_features=1024, out_features=2048, bias=False)),
 ('model.layers.7.self_attn.q_proj',
  Linear(in_features=1024, out_features=2048, bias=False)),
 ('model.layers.8.self_attn.q_proj',
  Linear(in_features=1024, out_features=2048, bias=False)),
 ('model.layers.9.self_attn.q_proj',
  Linear(in_features=1024, out_features=2048, bias=False)),
 ('model.layers.10.self_attn.q

In [ ]:
import pandas as pd

In [9]:
read_parquet = pd.read_parquet("~/Downloads/train-00000-of-00001.parquet", engine='fastparquet')

In [13]:
import numpy as np

In [17]:
(read_parquet.iloc[0]["map_array"])

'[[0. 0. 0. ... 0. 0. 0.]\n [0. 0. 0. ... 0. 0. 0.]\n [0. 0. 0. ... 0. 0. 0.]\n ...\n [0. 0. 0. ... 0. 0. 0.]\n [0. 0. 0. ... 0. 0. 0.]\n [0. 0. 0. ... 0. 0. 0.]]'